<a href="https://colab.research.google.com/github/ajzal4you/Master-Project-/blob/main/MultiOrgan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================
# Step 1: Environment Setup
# ================================
!pip install --quiet \
  numpy \
  pandas \
  matplotlib \
  seaborn \
  opencv-python \
  scikit-learn \
  tensorflow \
  keras \
  nibabel \
  medpy \
  shap \
  lime \
  kagglehub

import os, random, re, zipfile
from pathlib import Path
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import sklearn
import tensorflow as tf
import keras
import nibabel as nib
import shap
import lime
from medpy.io import load
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import Model as KModel

print("All packages imported successfully.")

# Reproducibility
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# ================================
# Step 2: Dataset Download and Preparation
# ================================
# Do not fail if kaggle.json is absent (kagglehub cache is fine)
os.makedirs("/root/.kaggle", exist_ok=True)
if os.path.exists("kaggle.json"):
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
else:
    print("kaggle.json not found; proceeding with kagglehub cache.")

import kagglehub

breast_path = kagglehub.dataset_download("uzairkhan45/breast-cancer-patients-mris")
print("Breast dataset downloaded to:", breast_path)

brain_path = kagglehub.dataset_download("nikhilroxtomar/brain-tumor-segmentation")
print("Brain dataset downloaded to:", brain_path)

kidney_path = kagglehub.dataset_download("orvile/t2-weighted-kidney-mri-segmentation")
print("Kidney dataset downloaded to:", kidney_path)

# ================================
# Step 3: Data Preprocessing
# ================================

# ---- 3A. Breast classification generators ----
breast_root = os.path.join(breast_path, "Breast Cancer Patients MRI's")
train_dir = os.path.join(breast_root, "train")
val_dir   = os.path.join(breast_root, "validation")

img_size = (224, 224)

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    class_mode='binary',
    batch_size=32,
    shuffle=True
)
val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=img_size,
    class_mode='binary',
    batch_size=32,
    shuffle=False
)

# SAMPLE VISUALS: Breast
xb, yb = next(train_gen)
plt.figure(figsize=(12, 6))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(xb[i])
    plt.title(f"Label: {int(yb[i])}")
    plt.axis("off")
plt.suptitle("Breast MRI – sample training images", y=1.02)
plt.tight_layout(); plt.show()

# ---- 3B. Brain PNG images + masks for segmentation ----
brain_img_folder  = os.path.join(brain_path, "images")
brain_mask_folder = os.path.join(brain_path, "masks")

brain_image_files = sorted(glob(os.path.join(brain_img_folder, "*.png")))
brain_mask_files  = sorted(glob(os.path.join(brain_mask_folder, "*.png")))
assert len(brain_image_files) == len(brain_mask_files) > 0, "Brain images/masks not found or mismatched."

img_names  = [os.path.basename(f) for f in brain_image_files]
mask_names = [os.path.basename(f) for f in brain_mask_files]
assert img_names == mask_names, "Brain image and mask filenames do not match."

def preprocess_brain(image_path, mask_path, size=(256, 256)):
    img  = cv2.imread(image_path, cv2.IMREAD_COLOR)          # BGR
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    img  = cv2.resize(img, size) / 255.0
    mask = cv2.resize(mask, size) / 255.0
    mask = np.expand_dims(mask, axis=-1)
    return img, mask

# Limit to 100 for speed
brain_images, brain_masks = [], []
for ip, mp in zip(brain_image_files[:100], brain_mask_files[:100]):
    im, mk = preprocess_brain(ip, mp)
    brain_images.append(im)
    brain_masks.append(mk)

brain_images = np.array(brain_images, dtype=np.float32)  # (N,256,256,3)
brain_masks  = np.array(brain_masks,  dtype=np.float32)  # (N,256,256,1)
print("brain_images shape:", brain_images.shape)
print("brain_masks  shape:", brain_masks.shape)

# SAMPLE VISUALS: Brain
plt.figure(figsize=(12, 6))
for i in range(5):
    plt.subplot(2, 5, i + 1);   plt.imshow(brain_images[i]);                 plt.title("MRI");   plt.axis("off")
    plt.subplot(2, 5, i + 6);   plt.imshow(brain_masks[i].squeeze(), 'gray'); plt.title("Mask");  plt.axis("off")
plt.suptitle("Brain tumour – sample image/mask pairs", y=1.02)
plt.tight_layout(); plt.show()

# ---- 3C. Kidney: attempt segmentation pairs, else FALLBACK to classification ----
def maybe_unzip_archives(root):
    root = Path(root)
    zips = list(root.rglob("*.zip"))
    for z in zips:
        try:
            with zipfile.ZipFile(z) as zf:
                zf.extractall(z.parent)
        except zipfile.BadZipFile:
            pass

maybe_unzip_archives(kidney_path)

_MASK_PAT = re.compile(r"(mask|label|seg|segmentation|annotation|ground.?truth|gt)", re.I)

def find_kidney_pairs(root):
    root = Path(root)
    pairs = []
    for dirpath, _, filenames in os.walk(root):
        d = Path(dirpath)
        nii_here = [f for f in filenames if f.lower().endswith((".nii", ".nii.gz"))]
        if not nii_here: continue
        masks = [d / f for f in nii_here if _MASK_PAT.search(f)]
        imgs  = [d / f for f in nii_here if not _MASK_PAT.search(f)]
        for sub in ["mask","masks","label","labels","annotation","annotations",
                    "ground_truth","gt","seg","segmentations"]:
            subdir = d / sub
            if subdir.exists():
                masks += list(subdir.rglob("*.nii")) + list(subdir.rglob("*.nii.gz"))
        for img in imgs:
            if not masks: continue
            def score(m):
                s_img = re.findall(r"\d+", img.stem.lower())
                s_m   = re.findall(r"\d+", Path(m).stem.lower())
                shared = len(set(s_img) & set(s_m))
                lcp = len(os.path.commonprefix([img.stem.lower(), Path(m).stem.lower()]))
                return (shared, lcp)
            m_best = sorted(masks, key=score, reverse=True)[0]
            pairs.append((str(img), str(m_best)))
    seen, uniq = set(), []
    for a, b in pairs:
        if (a, b) not in seen:
            uniq.append((a, b)); seen.add((a, b))
    return uniq

kidney_pairs = find_kidney_pairs(kidney_path)
kidney_mode = "segmentation" if len(kidney_pairs) > 0 else "classification"
print(f"Kidney mode resolved to: {kidney_mode.upper()}")

# ================================
# Step 4: Model Development
# ================================

# ---- 4A. Breast classification (MobileNetV2) ----
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights='imagenet'
)
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
predictions = Dense(1, activation='sigmoid')(x)

model_breast = Model(inputs=base_model.input, outputs=predictions)
model_breast.compile(optimizer=Adam(learning_rate=1e-4),
                     loss='binary_crossentropy',
                     metrics=['accuracy'])

# ---- 4B. Shared U-Net + Dice loss ----
def build_unet(input_shape=(128, 128, 1)):
    inputs = tf.keras.Input(input_shape)
    c1 = tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same')(inputs)
    c1 = tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same')(c1)
    p1 = tf.keras.layers.MaxPooling2D()(c1)

    c2 = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(p1)
    c2 = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(c2)
    p2 = tf.keras.layers.MaxPooling2D()(c2)

    c3 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(p2)
    c3 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(c3)
    p3 = tf.keras.layers.MaxPooling2D()(c3)

    c4 = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(p3)
    c4 = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(c4)

    u5 = tf.keras.layers.UpSampling2D()(c4)
    u5 = tf.keras.layers.Concatenate()([u5, c3])
    c5 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(u5)
    c5 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(c5)

    u6 = tf.keras.layers.UpSampling2D()(c5)
    u6 = tf.keras.layers.Concatenate()([u6, c2])
    c6 = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(u6)
    c6 = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(c6)

    u7 = tf.keras.layers.UpSampling2D()(c6)
    u7 = tf.keras.layers.Concatenate()([u7, c1])
    c7 = tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same')(u7)
    c7 = tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same')(c7)

    outputs = tf.keras.layers.Conv2D(1, 1, activation='sigmoid')(c7)
    return tf.keras.Model(inputs, outputs)

def dice_loss(y_true, y_pred):
    smooth = 1e-6
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (
        tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth
    )

# Brain U-Net (256x256x3)
model_brain = build_unet(input_shape=(256, 256, 3))
model_brain.compile(optimizer=Adam(learning_rate=1e-4), loss=dice_loss, metrics=['accuracy'])

# ================================
# Step 5: Training and Evaluation
# ================================

# ---- 5A. Train breast classifier ----
callbacks_breast = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ModelCheckpoint('model_breast.h5', save_best_only=True)
]
history_breast = model_breast.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=callbacks_breast,
    verbose=1
)

# ---- 5B. Train brain U-Net ----
callbacks_brain = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ModelCheckpoint('model_brain.h5', save_best_only=True)
]
history_brain = model_brain.fit(
    x=brain_images, y=brain_masks,
    validation_split=0.1,
    batch_size=8,
    epochs=5,
    callbacks=callbacks_brain,
    verbose=1
)

# ---- 5C. Kidney branch ----
if kidney_mode == "segmentation":
    # Build kidney U-Net and dataset from pairs
    model_kidney = build_unet(input_shape=(128, 128, 1))
    model_kidney.compile(optimizer=Adam(learning_rate=1e-4), loss=dice_loss, metrics=['accuracy'])

    def load_kidney_slices(img_path, mask_path, size=(128, 128), max_slices=None):
        img_vol  = nib.load(img_path).get_fdata()
        mask_vol = nib.load(mask_path).get_fdata()
        if img_vol.ndim == 4 and img_vol.shape[-1] == 1: img_vol = img_vol[..., 0]
        if mask_vol.ndim == 4 and mask_vol.shape[-1] == 1: mask_vol = mask_vol[..., 0]
        assert img_vol.shape == mask_vol.shape, f"Shape mismatch: {img_path} vs {mask_path}"
        vmax = np.max(img_vol) if np.max(img_vol) > 0 else 1.0
        img_vol = img_vol / vmax
        mask_vol = (mask_vol > 0).astype(np.float32)
        images, masks = [], []
        neg_budget = 10
        count = 0
        D = img_vol.shape[2]
        for i in range(D):
            sl_img  = img_vol[:, :, i]
            sl_mask = mask_vol[:, :, i]
            has_fg  = np.any(sl_mask > 0)
            if not has_fg:
                if neg_budget <= 0: continue
                neg_budget -= 1
            sl_img  = cv2.resize(sl_img.astype(np.float32), size)
            sl_mask = cv2.resize(sl_mask.astype(np.float32), size, interpolation=cv2.INTER_NEAREST)
            images.append(sl_img[..., None]); masks.append(sl_mask[..., None])
            count += 1
            if max_slices is not None and count >= max_slices: break
        return images, masks

    kidney_images, kidney_masks = [], []
    max_pairs_to_use = 6
    slices_per_pair  = 40
    for img_path, mask_path in kidney_pairs[:max_pairs_to_use]:
        xs, ys = load_kidney_slices(img_path, mask_path, size=(128, 128), max_slices=slices_per_pair)
        kidney_images.extend(xs); kidney_masks.extend(ys)

    kidney_images = np.array(kidney_images, dtype=np.float32)
    kidney_masks  = np.array(kidney_masks,  dtype=np.float32)

    # SAMPLE VISUALS: Kidney segmentation
    plt.figure(figsize=(10, 4))
    for i in range(3):
        idx = np.random.randint(0, kidney_images.shape[0])
        plt.subplot(2, 3, i + 1); plt.imshow(kidney_images[idx].squeeze(), cmap="gray"); plt.title("Kidney: image"); plt.axis("off")
        plt.subplot(2, 3, i + 4); plt.imshow(kidney_masks[idx].squeeze(),  cmap="gray"); plt.title("Kidney: mask");  plt.axis("off")
    plt.suptitle("Kidney – sample slices and masks", y=1.02)
    plt.tight_layout(); plt.show()

    Xk_train, Xk_val, yk_train, yk_val = train_test_split(
        kidney_images, kidney_masks, test_size=0.2, random_state=42
    )

    callbacks_kidney = [
        EarlyStopping(patience=3, restore_best_weights=True),
        ModelCheckpoint('model_kidney.h5', save_best_only=True)
    ]
    history_kidney = model_kidney.fit(
        Xk_train, yk_train,
        validation_data=(Xk_val, yk_val),
        epochs=5,
        batch_size=8,
        callbacks=callbacks_kidney,
        verbose=1
    )

else:
    # ---- 5C. Kidney branch (REPLACE ONLY THE else: block with this robust version) ----
    # FALLBACK: Kidney classification (CKD vs Healthy) from NIfTI slices
    import nibabel as nib

    # Reuse the same mask pattern used elsewhere
    _MASK_PAT = re.compile(r"(mask|label|seg|segmentation|annotation|ground.?truth|gt)", re.I)

    def is_nifti_valid(p):
        """Return True if nibabel can load the file, else False."""
        try:
            _ = nib.load(p)
            return True
        except Exception:
            return False

    def list_kidney_vols_as_classification(root):
        """Collect ONLY non-mask volumes; ignore masks and unreadable files."""
        root = Path(root)
        vols = []
        for sub, lab in [("CKD", 1), ("Healthy_Control", 0)]:
            for ext in ("*.nii", "*.nii.gz"):
                for p in root.joinpath(sub).rglob(ext):
                    name = p.name.lower()
                    if _MASK_PAT.search(name):
                        # exclude masks/labels/segmentations from classification
                        continue
                    if is_nifti_valid(str(p)):
                        vols.append((str(p), lab))
                    else:
                        print(f"[Kidney CLS] Skipping unreadable file: {p}")
        return vols

    vols = list_kidney_vols_as_classification(kidney_path)
    if not vols:
        raise RuntimeError("No usable kidney NIfTI volumes found for classification. Please inspect dataset contents.")

    # Split by subject (volume) to avoid leakage
    vol_paths, vol_labels = zip(*vols)
    Xtr_vols, Xva_vols, ytr_vols, yva_vols = train_test_split(
        vol_paths, vol_labels, test_size=0.2, random_state=42, stratify=vol_labels
    )

    def load_slices_for_cls(nii_path, label, size=(224,224), max_slices=40):
        """Load up to max_slices slices from a valid non-mask volume for classification."""
        vol = nib.load(nii_path).get_fdata()
        # Handle 4D volumes with a singleton last dim
        if vol.ndim == 4 and vol.shape[-1] == 1:
            vol = vol[..., 0]
        vmax = float(np.max(vol)) if np.max(vol) > 0 else 1.0
        vol = vol / vmax
        D = vol.shape[2]
        # uniform slice sampling for speed and balance
        idxs = np.linspace(0, D-1, min(max_slices, D)).astype(int).tolist()
        xs, ys = [], []
        for i in idxs:
            sl = cv2.resize(vol[:, :, i].astype(np.float32), size)
            sl3 = np.stack([sl, sl, sl], axis=-1)  # to 3 channels for MobileNet
            xs.append(sl3); ys.append(label)
        return xs, ys

    # Build train/val arrays (robust to occasional read errors)
    Xtr, ytr = [], []
    for p, y in zip(Xtr_vols, ytr_vols):
        try:
            xs, ys = load_slices_for_cls(p, y, size=(224,224), max_slices=40)
            Xtr.extend(xs); ytr.extend(ys)
        except Exception as e:
            print(f"[Kidney CLS] Skipping {p} due to read error: {e}")

    Xva, yva = [], []
    for p, y in zip(Xva_vols, yva_vols):
        try:
            xs, ys = load_slices_for_cls(p, y, size=(224,224), max_slices=40)
            Xva.extend(xs); yva.extend(ys)
        except Exception as e:
            print(f"[Kidney CLS] Skipping {p} due to read error: {e}")

    Xtr = np.asarray(Xtr, dtype=np.float32)
    Xva = np.asarray(Xva, dtype=np.float32)
    ytr = np.asarray(ytr, dtype=np.int32)
    yva = np.asarray(yva, dtype=np.int32)

    print("Kidney CLS train:", Xtr.shape, "val:", Xva.shape)
    assert len(Xtr) > 0 and len(Xva) > 0, "Kidney classification arrays are empty after filtering."

    # SAMPLE VISUALS: Kidney classification (one CKD and one Healthy volume)
    def show_kidney_cls_samples(vol_list, n_per_class=1):
        shown_ckd, shown_hc = 0, 0
        plt.figure(figsize=(10, 4))
        plot_idx = 1
        for path, lab in vol_list:
            if lab == 1 and shown_ckd < n_per_class:
                v = nib.load(path).get_fdata()
                if v.ndim == 4 and v.shape[-1] == 1: v = v[..., 0]
                mid = v.shape[2] // 2
                sl = v[:, :, mid].astype(np.float32)
                if sl.max() > 0: sl = sl / sl.max()
                plt.subplot(1, 2, plot_idx); plt.imshow(sl, cmap="gray"); plt.title("CKD sample"); plt.axis("off")
                shown_ckd += 1; plot_idx += 1
            if lab == 0 and shown_hc < n_per_class:
                v = nib.load(path).get_fdata()
                if v.ndim == 4 and v.shape[-1] == 1: v = v[..., 0]
                mid = v.shape[2] // 2
                sl = v[:, :, mid].astype(np.float32)
                if sl.max() > 0: sl = sl / sl.max()
                plt.subplot(1, 2, plot_idx); plt.imshow(sl, cmap="gray"); plt.title("Healthy sample"); plt.axis("off")
                shown_hc += 1; plot_idx += 1
            if shown_ckd >= n_per_class and shown_hc >= n_per_class:
                break
        plt.suptitle("Kidney classification – sample slices", y=1.02)
        plt.tight_layout(); plt.show()

    # show samples from the first items of each class
    show_kidney_cls_samples(vols, n_per_class=1)

    # Model: MobileNetV2 classifier (5 epochs)
    base_k = tf.keras.applications.MobileNetV2(
        input_shape=(224,224,3), include_top=False, weights='imagenet'
    )
    base_k.trainable = False
    xx = GlobalAveragePooling2D()(base_k.output)
    xx = Dropout(0.3)(xx)
    out = Dense(1, activation='sigmoid')(xx)
    model_kidney_cls = Model(inputs=base_k.input, outputs=out)
    model_kidney_cls.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])

    callbacks_kidney = [
        EarlyStopping(patience=3, restore_best_weights=True),
        ModelCheckpoint('model_kidney_cls.h5', save_best_only=True)
    ]
    history_kidney = model_kidney_cls.fit(
        Xtr, ytr, validation_data=(Xva, yva),
        epochs=5, batch_size=32, callbacks=callbacks_kidney, verbose=1
    )

# ---- 5D. Plot training curves ----
def plot_training(history, title=''):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    if 'val_accuracy' in history.history:
        plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{title} Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    if 'val_loss' in history.history:
        plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{title} Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
    plt.tight_layout(); plt.show()

plot_training(history_breast, title='Breast Cancer Classification')
plot_training(history_brain,  title='Brain Tumour Segmentation')
plot_training(history_kidney, title=('Kidney Segmentation' if kidney_mode=='segmentation' else 'Kidney Classification'))

# ---- 5E. Breast report + confusion matrix ----
y_pred_probs = model_breast.predict(val_gen)
y_pred = (y_pred_probs > 0.5).astype(int).reshape(-1)
y_true = val_gen.classes
print("\nBreast Cancer Classification Report:\n")
print(classification_report(y_true, y_pred))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')
plt.title("Breast Cancer – Confusion Matrix"); plt.show()

# ---- 5F. Brain segmentation metrics + overlays ----
def dice_score(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    return (2. * inter) / (K.sum(y_true_f) + K.sum(y_pred_f) + 1e-6)
def iou_score(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - inter
    return inter / (union + 1e-6)

y_pred_brain = model_brain.predict(brain_images)
y_pred_brain_bin = (y_pred_brain > 0.5).astype(np.float32)
mean_dice_brain = np.mean([
    dice_score(tf.convert_to_tensor(brain_masks[i], dtype=tf.float32),
               tf.convert_to_tensor(y_pred_brain_bin[i], dtype=tf.float32)).numpy()
    for i in range(len(brain_masks))
])
mean_iou_brain = np.mean([
    iou_score(tf.convert_to_tensor(brain_masks[i], dtype=tf.float32),
              tf.convert_to_tensor(y_pred_brain_bin[i], dtype=tf.float32)).numpy()
    for i in range(len(brain_masks))
])
print(f"Brain Tumour Dice Score: {mean_dice_brain:.4f}")
print(f"Brain Tumour IoU Score:  {mean_iou_brain:.4f}")

def show_segmentation_overlay(images, masks, predictions, title="Segmentation"):
    if tf.is_tensor(images): images = images.numpy()
    if tf.is_tensor(masks): masks = masks.numpy()
    if tf.is_tensor(predictions): predictions = predictions.numpy()
    plt.figure(figsize=(12, 4))
    for i in range(3):
        idx = np.random.randint(0, len(images))
        plt.subplot(3, 3, i*3 + 1); plt.imshow(images[idx].squeeze(), cmap='gray'); plt.title("Original"); plt.axis("off")
        plt.subplot(3, 3, i*3 + 2); plt.imshow(masks[idx].squeeze(),  cmap='gray'); plt.title("Ground Truth"); plt.axis("off")
        plt.subplot(3, 3, i*3 + 3); plt.imshow(predictions[idx].squeeze(), cmap='gray'); plt.title("Prediction"); plt.axis("off")
    plt.suptitle(title, fontsize=14); plt.tight_layout(); plt.show()

show_segmentation_overlay(brain_images, brain_masks, y_pred_brain_bin, title="Brain Tumour Segmentation")

# Kidney evaluation
if kidney_mode == "segmentation":
    y_pred_kidney = model_kidney.predict(Xk_val)
    y_pred_kidney_bin = (y_pred_kidney > 0.5).astype(np.float32)
    mean_dice_kidney = np.mean([
        dice_score(tf.convert_to_tensor(yk_val[i], dtype=tf.float32),
                   tf.convert_to_tensor(y_pred_kidney_bin[i], dtype=tf.float32)).numpy()
        for i in range(len(yk_val))
    ])
    mean_iou_kidney = np.mean([
        iou_score(tf.convert_to_tensor(yk_val[i], dtype=tf.float32),
                  tf.convert_to_tensor(y_pred_kidney_bin[i], dtype=tf.float32)).numpy()
        for i in range(len(yk_val))
    ])
    print(f"Kidney Dice Score: {mean_dice_kidney:.4f}")
    print(f"Kidney IoU Score:  {mean_iou_kidney:.4f}")
    show_segmentation_overlay(Xk_val, yk_val, y_pred_kidney_bin, title="Kidney Segmentation")
else:
    # Kidney classification report
    y_pred_probs_k = model_kidney_cls.predict(Xva)
    y_pred_k = (y_pred_probs_k > 0.5).astype(int).reshape(-1)
    print("\nKidney Classification Report (CKD vs Healthy):\n")
    print(classification_report(yva, y_pred_k))
    cmk = confusion_matrix(yva, y_pred_k)
    ConfusionMatrixDisplay(confusion_matrix=cmk).plot(cmap='Greens')
    plt.title("Kidney – Confusion Matrix (Classification)"); plt.show()

# ================================
# Step 6: Explainable AI (Grad-CAM on BREAST)
# ================================
xb_val, yb_val = next(val_gen)
breast_img = xb_val[0]
input_img = np.expand_dims(breast_img, 0)
last_conv_layer_name = "Conv_1"

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = KModel([model.inputs],
                        [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

heatmap = make_gradcam_heatmap(input_img, model_breast, last_conv_layer_name)
heatmap_resized = cv2.resize(heatmap, (224, 224))
heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
overlay = heatmap_colored * 0.4 + (breast_img * 255).astype("uint8")

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1); plt.imshow((breast_img * 255).astype("uint8")); plt.title("Original Breast Image"); plt.axis("off")
plt.subplot(1, 3, 2); plt.imshow(heatmap_resized, cmap='jet'); plt.title("Grad-CAM Heatmap"); plt.axis("off")
plt.subplot(1, 3, 3); plt.imshow(overlay.astype("uint8")); plt.title("Grad-CAM Overlay"); plt.axis("off")
plt.tight_layout(); plt.show()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 6.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 18.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━ 506.7/620.7 MB 3.6 MB/s eta 0:00:32